# Jev vs Claude Opus: clinical coherence and calibration benchmark (v5)

Everything lives in `MyDrive/jev_bench/`. Code for this version is in `code/v5/` and is checked against fixed SHA-256 hashes before anything runs. See `CHANGELOG.md` and each version's `CHANGES.md` for history.

**New in v5** (logged as deviations D4 and D5 in the pre-registration, 2026-09-27): adds Claude Opus 5 with thinking disabled (`opus5_nothink`) as a single-pass comparator alongside Claude Opus 5.5, which cannot run without thinking; logs token usage, including thinking tokens, for every Opus call; drops the Opus sampled arms from analysis; fixes top-1 tie-breaking so option order cannot flip a tied answer. Running this notebook adds only the new jobs; everything already in `results.jsonl` is kept and skipped.

**Secrets** (key icon, left sidebar, notebook access on): `OPENROUTER_API_KEY`, `ANTHROPIC_API_KEY`.

Run top to bottom. Sections 6 to 8 skip themselves until your data files are in `jev_bench/data/`.

## 1. Setup

In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys, shutil, hashlib
VERSION = 'v5'
WORK = '/content/drive/MyDrive/jev_bench'
CODE = f'{WORK}/code/{VERSION}'
D = f'{WORK}/data'
OUT = f'{WORK}/results.jsonl'
os.makedirs(D, exist_ok=True)

!pip -q install anthropic

for k in ('OPENROUTER_API_KEY', 'ANTHROPIC_API_KEY'):
    try:
        os.environ[k] = userdata.get(k)
        print(k, 'loaded')
    except Exception as e:
        print(k, 'MISSING:', e)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 16.0 MB/s eta 0:00:00
OPENROUTER_API_KEY loaded
ANTHROPIC_API_KEY loaded


## 2. Load and verify the harness code
Copies `code/v5/` into the session and refuses to continue if any file differs from the version's recorded hash.

In [2]:
EXPECTED = {'clients.py': '16472190f65fd6ec55e2888b89c8daf9176768a90ac5c191e13ebfb8a79f49d0', 'run.py': 'be272041e1e79549e93c8bdcf9c9fefa303443b561c2dd5d4c090b539a19c99e', 'analyze.py': '05466f1987577d6ee48006eb68c65b3d316a0b99d84b13355222b2f7e16bc6f9', 'ddxplus_to_cases.py': '0a2c9072ec480fe52e3eef4f420fcc2f37a71372d252c84a442e34a922912c51'}

def sha(path):
    return hashlib.sha256(open(path, 'rb').read()).hexdigest()

RUN = '/content/jev_bench'
os.makedirs(RUN, exist_ok=True)
bad = []
for f, h in EXPECTED.items():
    p = f'{CODE}/{f}'
    if not os.path.exists(p):
        bad.append(f'{f}: missing')
    elif sha(p) != h:
        bad.append(f'{f}: hash mismatch ({sha(p)[:12]} != {h[:12]})')
    else:
        shutil.copy(p, f'{RUN}/{f}')
if bad:
    raise SystemExit('Code check FAILED, paste this to Claude:\n' + '\n'.join(bad))
os.chdir(RUN)
sys.path.insert(0, RUN)
print('All', len(EXPECTED), 'files verified for', VERSION)

All 4 files verified for v5


## 3. Version control

`MyDrive/jev_bench/` is a git repository. `snapshot(label)` records a manifest (code hashes, data-file hashes, results hash), saves a snapshot of this notebook as it ran, commits, and sets `HARNESS_COMMIT` so every result row carries it. Data files are hashed but never committed. A plain timestamped copy of each manifest also goes to `versions/`.

In [3]:
import json, subprocess, datetime, glob

def _git(*args):
    return subprocess.run(['git', '-C', WORK, *args], capture_output=True, text=True)

GOOGLE_NATIVE = ('.gsheet', '.gdoc', '.gslides', '.gform', '.gdraw', '.gmap', '.gsite', '.gjam')

def _safe_sha(p):
    try:
        return sha(p)
    except OSError as e:
        return f'unreadable: {e.strerror}'

def _notebook_json():
    try:
        from google.colab import _message
        return _message.blocking_request('get_ipynb', request='', timeout_sec=30)['ipynb']
    except Exception as e:
        print('  (could not capture notebook contents:', e, ')')
        return None

def snapshot(label):
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    os.makedirs(f'{WORK}/runs', exist_ok=True)
    nbj = _notebook_json()
    if nbj:
        json.dump(nbj, open(f'{WORK}/runs/notebook_snapshot.ipynb', 'w'), indent=1)
    manifest = {'label': label, 'utc': stamp, 'version': VERSION,
                'code_sha256': {f: sha(f'{CODE}/{f}') for f in EXPECTED},
                'data_sha256': {os.path.relpath(p, WORK): _safe_sha(p)
                                for p in sorted(glob.glob(f'{D}/**/*', recursive=True))
                                if os.path.isfile(p) and not p.endswith(GOOGLE_NATIVE)},
                'results_sha256': sha(OUT) if os.path.exists(OUT) else None}
    json.dump(manifest, open(f'{WORK}/runs/MANIFEST.json', 'w'), indent=2)
    vdir = f'{WORK}/versions/{stamp}_{label}'
    os.makedirs(vdir, exist_ok=True)
    shutil.copy(f'{WORK}/runs/MANIFEST.json', vdir)

    if not os.path.exists(f'{WORK}/.git'):
        _git('init', '-q')
        _git('config', 'user.name', 'jev_bench')
        _git('config', 'user.email', 'jev_bench@localhost')
        _git('config', 'core.fileMode', 'false')
        open(f'{WORK}/.gitignore', 'w').write('data/\nversions/\n')
    _git('add', '-A')
    c = _git('commit', '-q', '-m', f'{VERSION} {label} {stamp}')
    head = _git('rev-parse', '--short', 'HEAD').stdout.strip()
    if head:
        os.environ['HARNESS_COMMIT'] = head
        print(f'snapshot "{label}": commit {head}' + ('' if c.returncode == 0 else ' (no changes)'))
    else:
        h = hashlib.sha256(''.join(manifest['code_sha256'].values()).encode()).hexdigest()[:10]
        os.environ['HARNESS_COMMIT'] = f'nogit-{h}'
        print(f'snapshot "{label}": git unavailable, using code hash {h}')

snapshot('setup')
print(_git('log', '--oneline', '-5').stdout)

snapshot "setup": commit f4845e8
f4845e8 v5 setup 20260927T161418Z
e765d1f v5 analysis 20260927T061044Z
b0e1890 v5 pre-run-opus5-nothink 20260927T051554Z
11f538f v5 pre-run-opus-verbalized 20260927T051551Z
5b48470 v5 pre-run-jev 20260927T051550Z



## 4. Offline smoke test (no API calls, no cost)

In [4]:
%%writefile example_options.json
{
  "toy_eye": {
    "options": {
      "retinal_detachment": "Retinal detachment",
      "posterior_vitreous_detachment": "Posterior vitreous detachment",
      "acute_angle_closure_glaucoma": "Acute angle-closure glaucoma",
      "central_retinal_artery_occlusion": "Central retinal artery occlusion",
      "bacterial_conjunctivitis": "Bacterial conjunctivitis",
      "anterior_uveitis": "Anterior uveitis",
      "corneal_abrasion": "Corneal abrasion",
      "dry_eye_disease": "Dry eye disease"
    },
    "distractors": {
      "ingrown_toenail": "Ingrown toenail",
      "benign_prostatic_hyperplasia": "Benign prostatic hyperplasia",
      "plantar_fasciitis": "Plantar fasciitis"
    }
  }
}


Writing example_options.json


In [5]:
%%writefile example_cases.csv
case_id,dataset,text,true_dx,paraphrase_1,paraphrase_2,paraphrase_3,distractors
t1,toy_eye,"Toy case: new flashes and a dark curtain coming over the vision in my right eye since yesterday.",retinal_detachment,"Toy case: since yesterday my right eye has flashing lights and a shade moving across.",,,ingrown_toenail;plantar_fasciitis
t2,toy_eye,"Toy case: sudden new floaters in my left eye, no change in vision otherwise, age 62.",posterior_vitreous_detachment,,,,ingrown_toenail
t3,toy_eye,"Toy case: severe eye pain, halos around lights, headache and vomiting tonight.",acute_angle_closure_glaucoma,,,,benign_prostatic_hyperplasia
t4,toy_eye,"Toy case: sudden painless total loss of vision in one eye an hour ago.",central_retinal_artery_occlusion,,,,plantar_fasciitis
t5,toy_eye,"Toy case: red eye with thick yellow discharge, lids stuck together this morning.",bacterial_conjunctivitis,,,,ingrown_toenail
t6,toy_eye,"Toy case: scratched my eye on a branch, now it waters and hurts to open.",corneal_abrasion,,,,ingrown_toenail


Writing example_cases.csv


In [6]:
%%writefile example_vague.csv
input_id,text
v1,"Toy input: my eye feels kind of off."
v2,"Toy input: something is not right, hard to explain."
v3,"Toy input: I just don't feel good today."


Writing example_vague.csv


In [7]:
!rm -f mock.jsonl
!python run.py --model mock --cases example_cases.csv --options example_options.json --vague example_vague.csv --out mock.jsonl
!python analyze.py mock.jsonl

91 jobs to run for mock (0 already done)
  25/91
  50/91
  75/91
# Results summary

Harness commit(s): f4845e8
Resolved model versions: mock=mock-0

## Output resolution (Amendment 1, descriptive)

- mock: exact-zero rate 0.0%; values on a 0.01 grid 0.0%; smallest nonzero 0.0011 (n=739)

## Token usage (v5, where recorded)

- Excluded from analysis: opus_sampled,opus_sampled_v2

## mock / toy_eye
- H0: 60/60 repeat pairs byte-identical; TVD noise floor (p95) = 0.0000; log-ratio floor = 0.0000
- H1: violation rate 100.0% [61.0%, 100.0%] (n=6); top-1 flip rate 80.0%; mean max-TVD 0.626; Wilcoxon p=0.0156
- H2: violation rate 100.0% [20.7%, 100.0%] (n=1); top-1 flip rate 0.0%; mean max-TVD 0.365
- D1 matched H1 (deviation): mean excess TVD perm minus repeat 0.4940 [0.4036, 0.5937]; excess top-1 flip rate +80.0%; two-sided Wilcoxon p=0.0312 (n=6)
- H3: violation rate 100.0% [61.0%, 100.0%] (n=6); mean distractor mass 0.0990; mean log-ratio shift 2.375
- H4: normalized entropy vague 0.680 v

## 5. Live checks (one call per model; a few cents)
Confirms Opus 5 accepts thinking disabled before the full run. Its usage line should show `thinking_tokens` of 0 or None and only `text` blocks.

In [8]:
import json
from collections import OrderedDict
from clients import get_client
opts = OrderedDict(sorted(json.load(open('example_options.json'))['toy_eye']['options'].items()))
state = 'New flashes and a dark curtain coming over the vision in my right eye since yesterday.'
for name in ('jev', 'opus_verbalized', 'opus5_nothink'):
    p, raw = get_client(name).classify(state, opts)
    print(f"{name} ({raw.get('resolved_model')}) usage: {raw.get('usage')}")
    for k, v in sorted(p.items(), key=lambda x: -x[1])[:3]:
        print(f'  {v:.3f}  {k}')

jev (typesafe/jev-1.13-20260917) usage: None
  0.910  retinal_detachment
  0.090  posterior_vitreous_detachment
  0.000  acute_angle_closure_glaucoma
opus_verbalized (claude-opus-5-5) usage: {'input_tokens': 323, 'output_tokens': 138, 'thinking_tokens': 0, 'stop_reason': 'end_turn', 'block_types': ['text']}
  0.870  retinal_detachment
  0.080  posterior_vitreous_detachment
  0.030  central_retinal_artery_occlusion
opus5_nothink (claude-opus-5) usage: {'input_tokens': 321, 'output_tokens': 141, 'thinking_tokens': 0, 'stop_reason': 'end_turn', 'block_types': ['text']}
  0.900  retinal_detachment
  0.076  posterior_vitreous_detachment
  0.008  central_retinal_artery_occlusion


## 6. Study data

Files go in `MyDrive/jev_bench/data/`: `cases.csv`, `options.json`, `vague.csv` for the vignette sets, and `release_test_patients.csv`, `release_evidences.json`, `release_conditions.json` for DDXPlus.

In [9]:
def have(*paths):
    missing = [p for p in paths if not os.path.exists(p)]
    if missing:
        print('Skipping: not uploaded yet ->', ', '.join(os.path.basename(m) for m in missing))
    return not missing

sh = get_ipython().system

if have(f'{D}/release_test_patients.csv', f'{D}/release_evidences.json', f'{D}/release_conditions.json'):
    sh(f'python ddxplus_to_cases.py --patients {D}/release_test_patients.csv --evidences {D}/release_evidences.json '
       f'--conditions {D}/release_conditions.json --n 1000 --out {D}/ddxplus_cases.csv --options-out {D}/ddxplus_options.json')
    snapshot('ddxplus-converted')

wrote 1000 cases, 49 options
snapshot "ddxplus-converted": commit 6eedf36


## 7. Runs
Only jobs not already in `results.jsonl` run. In v5 the Jev and Opus 5.5 cells find nothing new and finish immediately; the Opus 5 (thinking off) cell runs about 600 calls, roughly 40 to 60 minutes, printing progress every 25. A snapshot is committed before each run.

In [10]:
VIG = (f'{D}/cases.csv', f'{D}/options.json', f'{D}/vague.csv')
if have(*VIG):
    snapshot('pre-run-jev')
    sh(f'python run.py --model jev --cases {D}/cases.csv --options {D}/options.json --vague {D}/vague.csv --out {OUT} --max-calls 5000')

snapshot "pre-run-jev": commit 08ad80a
0 jobs to run for jev (600 already done)


In [11]:
if have(*VIG):
    snapshot('pre-run-opus-verbalized')
    sh(f'python run.py --model opus_verbalized --cases {D}/cases.csv --options {D}/options.json --vague {D}/vague.csv --out {OUT} --max-calls 5000')

snapshot "pre-run-opus-verbalized": commit 698cd34
0 jobs to run for opus_verbalized (600 already done)


In [12]:
# v5 (D4): Claude Opus 5, thinking disabled, single-pass comparator
if have(*VIG):
    snapshot('pre-run-opus5-nothink')
    sh(f'python run.py --model opus5_nothink --cases {D}/cases.csv --options {D}/options.json --vague {D}/vague.csv --out {OUT} --max-calls 5000')

snapshot "pre-run-opus5-nothink": commit 483889f
1 jobs to run for opus5_nothink (599 already done)


### Opus 5.5 thinking audit (optional, 45 calls)
The original Opus 5.5 run did not record token usage. This re-asks the 45 canonical questions once, only to measure how much Opus 5.5 thinks per answer, and saves the counts to `runs/opus55_thinking_audit.json`. These calls are not added to `results.jsonl` and do not change any result.

In [13]:
import csv as _csv
audit_path = f'{WORK}/runs/opus55_thinking_audit.json'
if have(f'{D}/cases.csv', f'{D}/options.json') and not os.path.exists(audit_path):
    o55 = get_client('opus_verbalized')
    base = OrderedDict(sorted(json.load(open(f'{D}/options.json'))['semigran45']['options'].items()))
    audit = {}
    for row in _csv.DictReader(open(f'{D}/cases.csv')):
        _, raw = o55.classify(row['text'], base)
        audit[row['case_id']] = raw.get('usage')
    json.dump(audit, open(audit_path, 'w'), indent=1)
    th = sorted((u or {}).get('thinking_tokens') or 0 for u in audit.values())
    print('Opus 5.5 thinking tokens: median', th[len(th)//2], 'min', th[0], 'max', th[-1])

In [14]:
if have(f'{D}/ddxplus_cases.csv', f'{D}/ddxplus_options.json'):
    snapshot('pre-run-ddxplus')
    for m in ('jev', 'opus_verbalized', 'opus5_nothink'):
        sh(f'python run.py --model {m} --cases {D}/ddxplus_cases.csv --options {D}/ddxplus_options.json --out {OUT} --conditions H0 --max-calls 1100')

snapshot "pre-run-ddxplus": commit 79850b7
1000 jobs to run for jev (600 already done)
  25/1000
  50/1000
  75/1000
  100/1000
  125/1000
  150/1000
  175/1000
  200/1000
  225/1000
  250/1000
  275/1000
  300/1000
  325/1000
  350/1000
  375/1000
  400/1000
  425/1000
  450/1000
  475/1000
  500/1000
  525/1000
  550/1000
  575/1000
  600/1000
  625/1000
  650/1000
  675/1000
  700/1000
  725/1000
  750/1000
  775/1000
  800/1000
  825/1000
  850/1000
  875/1000
  900/1000
  925/1000
  950/1000
  975/1000
  1000/1000
1000 jobs to run for opus_verbalized (600 already done)
  25/1000
  50/1000
  75/1000
  100/1000
  125/1000
  150/1000
  175/1000
  200/1000
  225/1000
  250/1000
  275/1000
  300/1000
  325/1000
  350/1000
  375/1000
  400/1000
  425/1000
  450/1000
  475/1000
  500/1000
  525/1000
  550/1000
  575/1000
  600/1000
  625/1000
  650/1000
  675/1000
  700/1000
  725/1000
  750/1000
  775/1000
  800/1000
  825/1000
  850/1000
  875/1000
  900/1000
  925/1000
  950/1000
  97

## 8. Analysis and final commit

In [15]:
if have(OUT):
    extra = f'--cases {D}/ddxplus_cases.csv' if os.path.exists(f'{D}/ddxplus_cases.csv') else ''
    sh(f'python analyze.py {OUT} {extra} > {WORK}/runs/summary.md')
    print(open(f'{WORK}/runs/summary.md').read())
    snapshot('analysis')
    print(_git('log', '--oneline', '-10').stdout)

# Results summary

Harness commit(s): 483889f, 54988ba, 79850b7, 8965c7b, 9ab7e87, b0e1890, ef4e15e
Resolved model versions: jev=typesafe/jev-1.13-20260917, opus5_nothink=claude-opus-5, opus_verbalized=claude-opus-5-5

## Output resolution (Amendment 1, descriptive)

- jev: exact-zero rate 93.1%; values on a 0.01 grid 99.1%; smallest nonzero 0.01 (n=71785)
- opus5_nothink: exact-zero rate 0.0%; values on a 0.01 grid 0.3%; smallest nonzero 0.0001 (n=71785)
- opus_verbalized: exact-zero rate 2.7%; values on a 0.01 grid 3.7%; smallest nonzero 0.0001 (n=71785)

## Token usage (v5, where recorded)

- opus5_nothink: n=1600; thinking tokens median 0 (IQR 0 to 0), share of calls with any thinking 0.0%; output tokens median 659
- opus_verbalized: n=1000; thinking tokens median 0 (IQR 0 to 197), share of calls with any thinking 41.2%; output tokens median 691
- Excluded from analysis: opus_sampled,opus_sampled_v2

## jev / ddxplus
- H6: n 1000; ECE 0.116; Brier 0.417; NLL_smoothed 1.179; NLL_raw